# AVA - Piper 'Vella' voice (resilient: auto-resume + checkpoint copy to Drive)

Run top to bottom. Cell 1 RESTARTS the runtime. At cell 3 approve the Drive-mount popup. Cell 5 AUTO-RESUMES from the newest checkpoint in Drive/ava_voice/vella_ckpts (or the base model on a fresh start), so every restart continues where the last left off. Cell 6 trains on LOCAL disk and copies each new checkpoint to Drive/ava_voice/vella_ckpts every 2 minutes. Run cell 6b to watch progress.

In [ ]:
# 1) Force Python 3.10 (condacolab) - RESTARTS the runtime
!pip install -q condacolab
import condacolab
condacolab.install_from_url('https://github.com/conda-forge/miniforge/releases/download/24.3.0-0/Miniforge3-24.3.0-0-Linux-x86_64.sh')

In [ ]:
# 2) Install Piper trainer + deps
import condacolab; condacolab.check()
!apt-get -qq install -y espeak-ng ffmpeg >/dev/null
!pip install -q piper-phonemize 2>&1 | tail -2
!git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip install -q -e . && pip install -q torchmetrics==0.11.4 'numpy<2' pydub soundfile six
!bash build_monotonic_align.sh
import torch; print('python ok | torch', torch.__version__, '| cuda', torch.cuda.is_available())
!python -c "import piper_train, piper_phonemize, six; print('PIPER OK')"

In [ ]:
# 3) Mount Drive + unpack the dataset from Drive
from google.colab import drive
drive.mount('/content/drive')
import os, glob, zipfile
ZIP = '/content/drive/MyDrive/ava_voice/vella_dataset.zip'
assert os.path.exists(ZIP), 'vella_dataset.zip missing in MyDrive/ava_voice/'
os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/extracted')
meta = glob.glob('/content/extracted/**/metadata.csv', recursive=True)[0]
!cp -r "{os.path.dirname(meta)}/." /content/dataset/
print('dataset ready: %d wavs' % len(glob.glob('/content/dataset/wavs/*.wav')))

In [ ]:
# 4) Piper preprocess (resamples to 22050)
!cd /content/piper/src/python && python -m piper_train.preprocess --language en-us --input-dir /content/dataset --output-dir /content/train --dataset-format ljspeech --single-speaker --sample-rate 22050

In [ ]:
# 5) Resume source: newest Drive checkpoint if present, else download the base model
import glob, os
saved = sorted(glob.glob('/content/drive/MyDrive/ava_voice/vella_ckpts/*.ckpt'), key=os.path.getmtime)
if saved:
    src = saved[-1]
    !cp "{src}" /content/base.ckpt
    print('RESUMING from saved checkpoint:', os.path.basename(src))
else:
    !wget -q --show-progress -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch=2164-step=1355540.ckpt"
    print('No saved checkpoint - starting fine-tune from BASE model')
!ls -lh /content/base.ckpt

In [ ]:
# 6) Fine-tune on LOCAL disk (reliable), background, copy each checkpoint to Drive every 2 min
import subprocess, threading, time, glob, shutil, os
DRIVE_CK = '/content/drive/MyDrive/ava_voice/vella_ckpts'
os.makedirs(DRIVE_CK, exist_ok=True)
cmd = ('cd /content/piper/src/python && python -m piper_train --dataset-dir /content/train '
       '--accelerator gpu --devices 1 --batch-size 12 --validation-split 0.05 --num-test-examples 2 '
       '--max_epochs 3000 --resume_from_checkpoint /content/base.ckpt --checkpoint-epochs 10 --precision 32 '
       '> /content/train.log 2>&1')
proc = subprocess.Popen(cmd, shell=True)
def copier():
    while proc.poll() is None:
        for c in sorted(glob.glob('/content/train/lightning_logs/**/checkpoints/*.ckpt', recursive=True)):
            dst = os.path.join(DRIVE_CK, os.path.basename(c))
            if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(c):
                try:
                    shutil.copy(c, dst); print('saved to Drive:', os.path.basename(c), flush=True)
                except Exception as e:
                    print('copy error:', e, flush=True)
        time.sleep(120)
threading.Thread(target=copier, daemon=True).start()
print('Training started in the BACKGROUND. Checkpoints save locally + copy to Drive every 2 min.')
print('Run cell 6b to watch progress.')

In [ ]:
# 6b) Watch progress (re-run any time)
import glob, os
!tail -n 15 /content/train.log
ck = sorted(glob.glob('/content/train/lightning_logs/**/checkpoints/*.ckpt', recursive=True))
dr = sorted(glob.glob('/content/drive/MyDrive/ava_voice/vella_ckpts/*.ckpt'))
print('--- local checkpoints:', [os.path.basename(c) for c in ck[-3:]])
print('--- Drive checkpoints :', [os.path.basename(c) for c in dr[-3:]])

In [ ]:
# 7) Export newest checkpoint -> ONNX (saved to Drive AND downloaded)
import glob, os
cands = glob.glob('/content/drive/MyDrive/ava_voice/vella_ckpts/*.ckpt') + glob.glob('/content/train/lightning_logs/**/checkpoints/*.ckpt', recursive=True)
cands = sorted(cands, key=os.path.getmtime)
assert cands, 'No checkpoint yet - let training run longer first.'
ck = cands[-1]; print('exporting', ck)
!cd /content/piper/src/python && python -m piper_train.export_onnx "{ck}" /content/drive/MyDrive/ava_voice/ava_vella.onnx
!cp /content/train/config.json /content/drive/MyDrive/ava_voice/ava_vella.onnx.json
from google.colab import files
files.download('/content/drive/MyDrive/ava_voice/ava_vella.onnx')
files.download('/content/drive/MyDrive/ava_voice/ava_vella.onnx.json')